# Milestone - Unit 12: tuning against race time

Unit 12 tuned Unit 11's agent against race time, and found that a search tuned on the training tracks alone overfits them.
`agents/unit-12.py` is `agents/unit-11.py` with exactly one module constant changed:

- `A_DEC = 5.5` (was 5.0): the only step a coordinate search over `MU_FRACTION`, `A_DEC` and `A_ACC` accepted on a **crash-aware** objective (the five training tracks at race seeds 0 and 1, plus `test-circuit` on Unit 11's three lowest-grip training cars 16, 18 and 8; any race that is not clean rejects a setting; Unit 1's rule accepts a step);
- it sits on that objective's edge: `A_DEC = 6.0` crashes, so Unit 11's margin rule would have shipped `unit-11.py` itself, and it is clean on 20 of Unit 11's 24 held-out perturbed cars where `unit-11.py` is clean on 24.

This milestone races design 013's gate for the unit (plan `roar-012`, premise **U12.5**) on the unit's **test track**, `test-circuit-seed2028`.
Its seed was fixed before the track was built or raced, no search or decision ever used it, and this is the race that measures the result:

- **U12.5** `unit-12.py` against `unit-11.py` on `test-circuit-seed2028`, with its own line (Unit 9's solver) and its probed grip (Unit 8's six probes, given as data), **3 laps**, race seeds 0-4: both clean in all five, and `noise_aware_better(unit-12's times, unit-11's times)`; medians 269.65 and 271.05 s;
- `check_submission(agents/unit-12.py)` passes, as shipped (`MPC_BUDGET_MS = 5.0`: the timing on, the one run in this unit that has it).

Every race uses racekit's carla-like car with its noise, `carla_like(noise=True)`, and `MPC_BUDGET_MS = None` (no wall clock inside a race, as in Units 10 and 11).
Tolerances (the plan's Gates): times within one tick (±0.05 s) of the pinned value.
A race is **clean** when it finished, with 0 collisions, and its max |cross-track error from its own line| over the driving rows is at most 2 m.
Every figure is a model number on racekit's carla-like car, not CARLA's.

## 1. Set-up: the test track's line and grip

The milestone runs from `projects/racekit/milestones/`, so the agents are one directory up.
The test track's line is solved here with Unit 9's solver; its grip is Unit 8's six probes on it (plan `roar-012`, premise U12.0: section 0's probe finished without sliding, so its value is a lower bound, used as measured).

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.check_submission import check_submission
from racekit.measure import noise_aware_better, repeat_races
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("..") / "agents"
UNIT_11, UNIT_12 = AGENTS / "unit-11.py", AGENTS / "unit-12.py"
TEST = "test-circuit-seed2028"  # Unit 12's test track
TEST_MU_G = (9.01, 9.13, 8.63, 9.38, 8.20, 10.09)  # m/s^2: Unit 8's six grip probes on it (premise U12.0)
SEEDS = (0, 1, 2, 3, 4)
TICK = 0.05 + 1e-9  # s: the plan's tolerance, one tick
CLEAN_ERROR = 2.0  # m
CAP = 450.0  # s
NO_TIMING = {"MPC_BUDGET_MS": None}
workdir = Path(mkdtemp(prefix="roar-milestone-12-"))


# Unit 9's line solver: a cubic B-spline of offsets, the smoothest line by bounded least squares.
def spline_basis(n, step=10):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track, margin=2.5, pin=30):
    x, y = track.x, track.y
    nx, ny = normals(x, y)
    basis = spline_basis(len(x))
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    rows, columns = basis.shape
    index = np.arange(rows)
    pinned = (index < pin) | (index >= rows - pin)
    high = np.zeros(columns)
    for j in range(columns):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(pinned[support]) > 0 else np.min(track.half_width[support] - margin)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x


test_track = load_track(TEST)
test_control = solve_line(test_track)
test_offsets = spline_basis(len(test_track)) @ test_control
print(f"{TEST}: {len(test_track)} waypoints, its line {len(test_control)} control values, "
      f"at most {np.max(np.abs(test_offsets)):.2f} m from the centreline")
assert len(test_track) == 1314 and len(test_control) == 131  # U12.0


def on_test_track(agent, name):
    # The agent with the test track's line and grip written in, timing off.
    constants = {"LINE_CONTROL": tuple(float(value) for value in test_control), "MU_G": TEST_MU_G, **NO_TIMING}
    return write_variant(agent, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def line_error(result):
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = test_offsets[(telemetry["progress"][driving] + 1) % len(test_offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result):
    return result.finished and result.collisions == 0 and line_error(result) <= CLEAN_ERROR


def five_races(agent, name):
    # Three laps of the test track, race seeds 0-4, the nominal carla-like car with its noise.
    return repeat_races(on_test_track(agent, name), TEST, n=5, seeds=SEEDS, car_factory=lambda: carla_like(noise=True),
                        laps=3, max_seconds=CAP)


def report(name, runs):
    for seed, result in zip(SEEDS, runs.results):
        print(f"{name} seed {seed}: {result.total_time:.2f} s, clean {clean(result)}, {result.collisions} collisions, "
              f"{line_error(result):.2f} m from its line")
    print(f"{name}: median {runs.median:.2f} s, spread {runs.spread:.2f} s")

## 2. U12.5: `unit-12.py` on the test track

Five races, 3 laps each (about 30 s).

In [ ]:
twelve = five_races(UNIT_12, "u12-test")
report("unit-12.py", twelve)
assert all(clean(result) for result in twelve.results)  # U12.5: 5/5 clean
assert abs(twelve.median - 269.65) <= TICK  # U12.5

## 3. U12.5: `unit-11.py` on the test track, and the gate

The same five races for the hand-tuned agent, then Unit 1's rule.

In [ ]:
eleven = five_races(UNIT_11, "u11-test")
report("unit-11.py", eleven)
assert all(clean(result) for result in eleven.results)  # U12.5: 5/5 clean
assert abs(eleven.median - 271.05) <= TICK  # U12.5

gate = noise_aware_better(twelve.times, eleven.times)
print(f"unit-12.py is {eleven.median - twelve.median:.2f} s faster (median); the larger spread is "
      f"{max(twelve.spread, eleven.spread):.2f} s; Unit 1's rule: unit-12.py beats unit-11.py: {gate}")
print("seed  unit-12.py  unit-11.py")
for seed, a, b in zip(SEEDS, twelve.times, eleven.times):
    print(f"{seed:4d}  {a:10.2f}  {b:10.2f}")
assert gate  # U12.5: design 013's gate for Unit 12

## 4. `check_submission`, with the timing on

The shipped file, exactly as shipped (`MPC_BUDGET_MS = 5.0`): a Python 3.8 deny-list over its syntax, its imports (numpy, `roar_py_interface` and the Python 3.8 standard library only), the official shape (`RoarCompetitionSolution`, `async def initialize` and `step`), and one race on `test-circuit` with `step()`'s 99th percentile at most 5 ms.
It is the one run in this unit with the clock inside the race, so its own race time is not pinned; run it when the machine is not busy.

In [ ]:
result = check_submission(UNIT_12)
print(result)
assert result.ok, result.findings

## Unit 12, pinned

| `test-circuit-seed2028`, 3 laps, race seeds 0-4, `carla_like(noise=True)` | `unit-11.py` (hand-tuned) | `unit-12.py` (tuned) |
|---|---|---|
| clean | 5/5 | 5/5 |
| median (spread) | 271.05 s (0.20) | **269.65 s** (0.10) |
| Unit 1's rule | | `unit-12.py` beats `unit-11.py` |

The routed premise (`tests/test_racekit_unit12_premise.py`) races the rest: the same 1.35 s on the validation track `test-circuit-seed2026` (280.65 against 282.00 s, U12.5a), the crash-aware search's verdict and its edge (U12.6), and the cost: `unit-12.py` finishes all 24 of Unit 11's held-out perturbed cars with 0 collisions, but only 20 of them clean (U12.7).
Which of the two races on race day is Unit 13's decision.